In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
from src.config import (
    load_config,
    get_embedding_model,
    get_reranker_model,
    get_top_k,
)

from src.knowledge_base import KnowledgeBaseService
from src.embeddings import EmbeddingService
from src.faiss_index import FaissIndexService
from src.retriever import RetrieverService
from src.reranker import RerankerService
from src.rag_pipeline import RAGPipeline

In [3]:
DATA_DIR = Path("../data/raw")
train = pd.read_csv(DATA_DIR / "train.csv")
config = load_config()

knowledge_base_service = KnowledgeBaseService()
knowledge_base_service.build(train)

embedding_service = EmbeddingService(
    get_embedding_model(config)
)

faiss_service = FaissIndexService()

faiss_service.load(
    PROJECT_ROOT /
    "outputs" /
    "faiss" /
    "knowledge_base.faiss"
)

retriever_service = RetrieverService(
    embedding_service,
    faiss_service,
)

reranker_service = RerankerService(
    get_reranker_model(config)
)

2026-07-13 20:47:42,957 | INFO | src.knowledge_base | Building Knowledge Base...
2026-07-13 20:47:42,972 | INFO | src.knowledge_base | Knowledge Base built successfully with 2000 documents.
2026-07-13 20:47:42,972 | INFO | src.embeddings | Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-13 20:47:49,955 | INFO | src.embeddings | Embedding model loaded.
2026-07-13 20:47:49,958 | INFO | src.faiss_index | FAISS index loaded successfully.
2026-07-13 20:47:49,959 | INFO | src.reranker | Loading Cross Encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

2026-07-13 20:48:00,062 | INFO | src.reranker | Cross Encoder loaded successfully.


In [4]:
rag_pipeline = RAGPipeline(
    knowledge_base_service,
    retriever_service,
    reranker_service,
)

In [5]:
query = "What is machine learning?"

results = rag_pipeline.retrieve_context(
    query,
    get_top_k(config),
)

2026-07-13 20:48:00,076 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-13 20:48:00,077 | INFO | src.retriever | Embedding query.
2026-07-13 20:48:00,077 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-13 20:48:00,172 | INFO | src.embeddings | Embedding generation completed.
2026-07-13 20:48:00,172 | INFO | src.retriever | Searching FAISS index.
2026-07-13 20:48:00,174 | INFO | src.retriever | Retrieved 10 documents.
2026-07-13 20:48:00,175 | INFO | src.reranker | Reranking 10 retrieved documents.
2026-07-13 20:48:00,219 | INFO | src.reranker | Reranking completed.
2026-07-13 20:48:00,220 | INFO | src.rag_pipeline | RAG pipeline completed.


In [6]:
for rank, result in enumerate(results, start=1):

    print(f"Rank {rank}")
    print(f"Score : {result['score']:.4f}")
    print(result["document"][:200])
    print("-" * 80)

Rank 1
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 2
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 3
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 4
Score : -6.0902
A device used to demonstrate a neuro-inspired circuit that shows long-term potentiation for learning and inactivity-based forgetting.
--------------------------------------------------------------------------------
Rank 5
Score : -6.0902
A device used to demonstr

In [7]:
assert len(results) == get_top_k(config)

assert results[0]["score"] >= results[-1]["score"]

In [8]:
test = pd.read_csv(DATA_DIR / "test.csv")
row = test.iloc[0]

prediction = rag_pipeline.predict(
    row,
    top_k=3,
)

print(prediction)

2026-07-13 20:48:00,246 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-13 20:48:00,246 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-13 20:48:00,247 | INFO | src.retriever | Embedding query.
2026-07-13 20:48:00,247 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-13 20:48:00,398 | INFO | src.embeddings | Embedding generation completed.
2026-07-13 20:48:00,399 | INFO | src.retriever | Searching FAISS index.
2026-07-13 20:48:00,399 | INFO | src.retriever | Retrieved 3 documents.
2026-07-13 20:48:00,399 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-13 20:48:00,614 | INFO | src.reranker | Reranking completed.
2026-07-13 20:48:00,615 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-13 20:48:00,615 | INFO | src.rag_pipeline | Prediction completed.


['A', 'A', 'A']


In [9]:
predictions = []

for _, row in test.iterrows():

    prediction = rag_pipeline.predict(
        row,
        top_k=3,
    )

    predictions.append(
        " ".join(prediction)
    )

2026-07-13 20:48:35,359 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-13 20:48:35,362 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-13 20:48:35,362 | INFO | src.retriever | Embedding query.
2026-07-13 20:48:35,362 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-13 20:48:35,416 | INFO | src.embeddings | Embedding generation completed.
2026-07-13 20:48:35,416 | INFO | src.retriever | Searching FAISS index.
2026-07-13 20:48:35,417 | INFO | src.retriever | Retrieved 3 documents.
2026-07-13 20:48:35,417 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-13 20:48:35,436 | INFO | src.reranker | Reranking completed.
2026-07-13 20:48:35,437 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-13 20:48:35,437 | INFO | src.rag_pipeline | Prediction completed.
2026-07-13 20:48:35,438 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-13 20:48:35,438 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-13 20:48:35,438 | INFO | src.retriever | Embedding query.
2026-07-13 20:48:35,438 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-13 20:48:35,487 | INFO | src.embeddings | Embedding generation completed.
2026-07-13 20:48:35,488 | INFO | src.retriever | Searching FAISS index.
2026-07-13 20:48:35,488 | INFO | src.retriever | Retrieved 3 documents.
2026-07-13 20:48:35,488 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-13 20:48:35,507 | INFO | src.reranker | Reranking completed.
2026-07-13 20:48:35,508 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-13 20:48:35,508 | INFO | src.rag_pipeline | Prediction completed.
2026-07-13 20:48:35,509 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-13 20:48:35,509 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-13 20:48:35,509 | INFO | src.retriever | Embedding query.
2026-07-13 20:48:35,510 | INFO | src.embeddings | Generating embeddings for 1 documents.


...

2026-07-13 20:48:51,738 | INFO | src.embeddings | Embedding generation completed.
2026-07-13 20:48:51,739 | INFO | src.retriever | Searching FAISS index.
2026-07-13 20:48:51,739 | INFO | src.retriever | Retrieved 3 documents.
2026-07-13 20:48:51,739 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-13 20:48:51,748 | INFO | src.reranker | Reranking completed.
2026-07-13 20:48:51,749 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-13 20:48:51,749 | INFO | src.rag_pipeline | Prediction completed.
2026-07-13 20:48:51,749 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-13 20:48:51,750 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-13 20:48:51,750 | INFO | src.retriever | Embedding query.
2026-07-13 20:48:51,750 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-13 20:48:51,759 | INFO | src.embeddings | Embedding generation completed.
2026-07-13 20:48:51,759 | INFO | src.retriever | Searching FAISS index.
2026-07-13 20:48:51,759 | INFO | src.retriever | Retrieved 3 documents.
2026-07-13 20:48:51,760 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-13 20:48:51,767 | INFO | src.reranker | Reranking completed.
2026-07-13 20:48:51,768 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-13 20:48:51,768 | INFO | src.rag_pipeline | Prediction completed.
2026-07-13 20:48:51,768 | INFO | src.rag_pipeline | Predicting answer options.
2026-07-13 20:48:51,768 | INFO | src.rag_pipeline | Executing RAG pipeline.
2026-07-13 20:48:51,769 | INFO | src.retriever | Embedding query.
2026-07-13 20:48:51,769 | INFO | src.embeddings | Generating embeddings for 1 documents.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-07-13 20:48:51,777 | INFO | src.embeddings | Embedding generation completed.
2026-07-13 20:48:51,778 | INFO | src.retriever | Searching FAISS index.
2026-07-13 20:48:51,778 | INFO | src.retriever | Retrieved 3 documents.
2026-07-13 20:48:51,778 | INFO | src.reranker | Reranking 3 retrieved documents.
2026-07-13 20:48:51,786 | INFO | src.reranker | Reranking completed.
2026-07-13 20:48:51,786 | INFO | src.rag_pipeline | RAG pipeline completed.
2026-07-13 20:48:51,786 | INFO | src.rag_pipeline | Prediction completed.


In [10]:

submission = test[["id"]].copy()

submission["prediction"] = predictions

submission.head()

,id,prediction
0,1,A A A
1,2,
2,3,B B B
3,4,E E E
4,5,C C C


In [11]:
submission.to_csv(
    PROJECT_ROOT /
    "outputs" /
    "submission.csv",
    index=False,
)